In [1]:
from anthropic import Anthropic
from dotenv import load_dotenv
from building_with_the_claude_api import add_assistant_message, add_user_message, chat, Effort

load_dotenv()
client = Anthropic()
system_prompt = """Behave as if you are explain to a history professor with no STEM background.
Their son wants to major in one of those fields."""

In [2]:
messages = []

while True:
    user_input = input("> ")
    print("---")
    print(user_input)

    add_user_message(messages=messages, text=user_input)
    response = chat(messages=messages, client=client, system_prompt=system_prompt, effort=Effort.LOW)
    add_assistant_message(messages=messages, text=response)
    print("---")
    print(response)


KeyboardInterrupt: Interrupted by user

In [ ]:
# Streaming entire message events
messages = []
add_user_message(messages=messages, text="Generate one sentence ML eng portfolio project idea.")

request = {
    "model": "claude-sonnet-5",
    "max_tokens": 1024,
    "messages": messages,
    "output_config": {"effort": Effort.MEDIUM},
    "stream": True
}

stream = client.messages.create(**request)

for message in stream:
    print(message)

In [ ]:
# Streaming text response
messages = []
add_user_message(messages=messages, text="Generate one sentence ML eng portfolio project idea.")

request = {
    "model": "claude-sonnet-5",
    "max_tokens": 1024,
    "messages": messages,
    "output_config": {"effort": Effort.MEDIUM},
}

with client.messages.stream(**request) as stream:
    for text in stream.text_stream:
        print(text, end="")

print("\n---")
stream.get_final_message()

In [ ]:
# Structured data: Claude wrap the json with some sort of explaination
messages = []

add_user_message(messages=messages, text="Generate a very short event bridge rule as json")
text = chat(messages=messages, client=client)
text


In [2]:
# Structured data: force a JSON-only response via structured outputs
# (claude-sonnet-5 dropped assistant message prefill, so the classic
# "```json" prefill trick no longer works -- use output_config.format instead)
import json
messages = []

add_user_message(messages=messages, text="Generate a very short event bridge rule as json")

event_bridge_rule_schema = {
    "type": "object",
    "properties": {
        "source": {"type": "array", "items": {"type": "string"}},
        "detail-type": {"type": "array", "items": {"type": "string"}},
        "detail": {
            "type": "object",
            "properties": {"state": {"type": "array", "items": {"type": "string"}}},
            "required": ["state"],
            "additionalProperties": False,
        },
    },

    "required": ["source", "detail-type", "detail"],
    "additionalProperties": False,
}

text = chat(messages=messages, client=client, effort=Effort.LOW, json_schema=event_bridge_rule_schema)

json.loads(text)


{'source': ['aws.ec2'],
 'detail-type': ['EC2 Instance State-change Notification'],
 'detail': {'state': ['running']}}